<a href="https://colab.research.google.com/github/mteuslms/AI-ML-projects/blob/main/LLMs_and_Transformers/Spam_or_Ham.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bonus Assignment
The Goal: A binary classifier that flags incoming text messages or emails as Legitimate (Ham) or Malicious (Spam).

Why it’s highly testable: We can easily test it by typing something like "URGENT: Your Wells Fargo account has been suspended. Click here to verify: http://bit.ly/scam". If it predicts Spam, you know for a fact it worked. If you type "Hey man, are we still on for lunch at 12?" and it predicts Legitimate, you know it's accurate.

The Data: The UCI SMS Spam Collection dataset (about 5,500 real text messages, perfectly labeled).

In [ ]:
#using fastai
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import csv

from fastai.text.all import *
import torch.nn as nn

# checking for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# 1. Download the raw dataset directly into Pandas
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'text'])

# 2. Map the text labels to binary integers
# Let's make Spam = 1 (The threat we want to detect) and Ham = 0 (Safe)
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

# 3. Check the balance and the first few rows
print(df['label'].value_counts())
print("\nFirst 3 messages:")
print(df.head(3))

cuda
label
0    4825
1     747
Name: count, dtype: int64

First 3 messages:
   label  \
0      0   
1      0   
2      1   

                                                                                                                                                          text  
0                                              Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...  
1                                                                                                                                Ok lar... Joking wif u oni...  
2  Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's  


Building model

In [ ]:
from fastai.text.all import *
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# SMS messages are short, so we only need to look at ~60 words max
wl = 60

# 1. Prepare the DataLoaders
dls = TextDataLoaders.from_df(
    df,
    text_col='text',        # The column containing the text message
    label_col='label',      # 0 for Ham, 1 for Spam
    valid_pct=0.2,          #this parameter splits the data between training and validation
    bs=64,                  #batch size is 64 messages at a time
    seq_len=wl,             #Sequence Length, set to 60 to limit how many words the model will read per message
    seed=17,                #random seed, ensures that the 80/20 split is the same random split every time I run the code
    device=device           #tells FastAI where to physically send the data on my environment (Our case is the T4 gpu)
)

# 2. Build the exact same Neural Network from your lab
embed_size = 300
vocab_size = len(dls.vocab[0])
print(f"\nFastAI Vocab Size: {vocab_size}")

model = nn.Sequential(
    nn.Embedding(vocab_size, embed_size),
    nn.AdaptiveAvgPool2d((1, embed_size)),
    nn.Flatten(),
    nn.Linear(embed_size, 64),
    nn.ReLU(),
    nn.Linear(64, 256),
    nn.ReLU(),
    nn.Linear(256, 1024),
    nn.ReLU(),
    nn.Linear(1024, 64),
    nn.ReLU(),
    nn.Linear(64, 2)
)

# 3. Set up the Learner
learn = Learner(dls, model, loss_func=nn.CrossEntropyLoss(), metrics=accuracy)

# Colab Progress Bar Fix
from fastai.callback.progress import ProgressCallback
learn.remove_cb(ProgressCallback)

print("Spam Filter is ready to train!")


FastAI Vocab Size: 2944
Spam Filter is ready to train!


Training Model

In [ ]:
# Train for 10 epochs
learn.fit(n_epoch=10, lr=0.001)

# Validate
train_loss, train_accuracy = learn.validate(dl=dls.train)
valid_loss, valid_accuracy = learn.validate(dl=dls.valid)

print(f"Train Accuracy: {train_accuracy}")
print(f"Validation Accuracy: {valid_accuracy}")



[0, 0.2701934576034546, 0.276984840631485, 0.8698384165763855, '00:05']
[1, 0.14967858791351318, 0.2582801580429077, 0.9605026841163635, '00:04']
[2, 0.09000123292207718, 0.18654631078243256, 0.9578096866607666, '00:05']
[3, 0.057724110782146454, 0.08962255716323853, 0.9658886790275574, '00:04']
[4, 0.08256078511476517, 0.12407802045345306, 0.9434470534324646, '00:04']
[5, 0.04738539829850197, 0.162689208984375, 0.9667863845825195, '00:05']
[6, 0.03743365406990051, 0.10276211053133011, 0.9667863845825195, '00:04']
[7, 0.06164351850748062, 0.1511489897966385, 0.9676840305328369, '00:05']
[8, 0.03720368817448616, 0.08984041959047318, 0.9703770279884338, '00:04']
[9, 0.02953486144542694, 0.09618960320949554, 0.9703770279884338, '00:04']
Train Accuracy: 0.9923007488250732
Validation Accuracy: 0.9703770279884338


In [ ]:
#testing the model
import torch

# 1. The 10 Custom Test Messages
test_messages = [
    # --- LEGITIMATE HAM (Class 0) ---
    "Hey, are we still meeting up for lunch at noon?",
    "Can you send me the PDF of the lecture slides when you get a chance?",
    "The meeting got pushed back to 3 PM. See you then in the conference room.",
    "I'm picking up groceries, do we need more milk or eggs?",
    "Just finished running the script, the output looks correct now.",

    # --- MALICIOUS SPAM (Class 1) ---
    "URGENT: Your Wells Fargo account has been locked due to suspicious activity. Click here to verify your identity: http://secure-update-login.com",
    "Congratulations! You've been selected to win a free $1000 Walmart Gift Card. Reply WIN to claim your prize now!",
    "FedEx: Your package delivery failed. Please update your shipping address and pay the $2.99 redelivery fee here: https://fedex-tracking-alert.info",
    "IRS ALERT: You have an unpaid tax balance of $1,450. A warrant for your arrest will be issued in 24 hours. Call 1-800-555-0199 immediately.",
    "Your car's extended warranty is about to expire! Press 1 to speak with an agent or visit http://auto-warranty-renew.com"
]

print("🛡️ RUNNING ZERO-DOUBT SPAM DETECTOR...\n" + "="*50)

# 2. Loop through and predict each message
for i, msg in enumerate(test_messages):
    # Process the text through FastAI's pipeline
    tokens = dls.tokenizer(msg)
    numericalized_tokens = dls.numericalize(tokens)
    test_dl = dls.test_dl([numericalized_tokens], with_labels=False)

    # Get prediction
    preds, _ = learn.get_preds(dl=test_dl)
    pred_class_idx = torch.argmax(preds, dim=-1).item()
    pred_class = dls.vocab[1][pred_class_idx]

    # 3. Format the Output
    if pred_class == 1:
        print(f"[{i+1}/10] 🚨 SPAM DETECTED (Class 1)")
    else:
        print(f"[{i+1}/10] ✅ SAFE HAM (Class 0)")

    print(f"Text: '{msg}'\n{'-'*50}")

🛡️ RUNNING ZERO-DOUBT SPAM DETECTOR...
[1/10] ✅ SAFE HAM (Class 0)
Text: 'Hey, are we still meeting up for lunch at noon?'
--------------------------------------------------
[2/10] ✅ SAFE HAM (Class 0)
Text: 'Can you send me the PDF of the lecture slides when you get a chance?'
--------------------------------------------------
[3/10] ✅ SAFE HAM (Class 0)
Text: 'The meeting got pushed back to 3 PM. See you then in the conference room.'
--------------------------------------------------
[4/10] ✅ SAFE HAM (Class 0)
Text: 'I'm picking up groceries, do we need more milk or eggs?'
--------------------------------------------------
[5/10] ✅ SAFE HAM (Class 0)
Text: 'Just finished running the script, the output looks correct now.'
--------------------------------------------------
[6/10] 🚨 SPAM DETECTED (Class 1)
Text: 'URGENT: Your Wells Fargo account has been locked due to suspicious activity. Click here to verify your identity: http://secure-update-login.com'
------------------------------